# Bölüm 11: Transformer'ı İnşa Etmek

> "Biz tekrar tekrar yaptığımız şeyleriz. Öyleyse mükemmellik bir eylem değil, bir alışkanlıktır."
> — **Aristoteles**, Filozof

---

## Neler Öğreneceksiniz

- Transformer bloklarını nasıl tam bir dil modeline dönüştüreceğiz
- Model deneylemeyi kolaylaştıran konfigürasyon deseni
- Dil modelleme başlığının ne yaptığı ve neden buna ihtiyacımız olduğu
- Ağırlık bağlama: 38 milyon parametreyi kaydeden zarif bir hile
- Modelinizi eğitmeden önce doğrulamak için temel sağlık kontrolleri
- Önceden eğitilmiş GPT-2 ağırlıklarını mimarinize nasıl yükleyeceğiz

---

## Kurulum

Önce gerekli paketleri yükleyelim:

In [ ]:
# Gerekli paketleri yükle
!pip install -q torch transformers

In [ ]:
# ===== İÇE AKTARMALAR =====
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from transformers import AutoTokenizer

# Cihazı ayarla
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan cihaz: {device}")

## 1. Bölüm 10'dan Bileşenler

Önce, Bölüm 10'da oluşturduğumuz bileşenleri getirelim: `MultiHeadAttention`, `FeedForward` ve `TransformerBlock`.

In [ ]:
# ===== ÇOK BAŞLIKLI DİKKAT (Bölüm 10'dan) =====

class MultiHeadAttention(nn.Module):
    """Verimli çok başlıklı dikkat (tüm başlıkları birlikte yığınlar)."""
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model, num_heads ile bölünebilir olmalıdır"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        
        # Birleştirilmiş QKV projeksiyon
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape
        
        # Q, K, V'ye yansıt
        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        
        # Ölçeklendirilmiş nokta-çarpım dikkati
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        
        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Ağırlıklı toplam ve birleştir
        attn_output = attn_weights @ V
        attn_output = attn_output.transpose(1, 2).reshape(batch, seq, d_model)
        
        return self.out_proj(attn_output), attn_weights

print("MultiHeadAttention tanımlandı!")

In [ ]:
# ===== İLERİ BESLEMELI AĞ (Bölüm 10'dan) =====

class FeedForward(nn.Module):
    """Konum-bazlı ileri beslemeli ağ."""
    
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("FeedForward tanımlandı!")

In [ ]:
# ===== TRANSFORMER BLOĞU (Bölüm 10'dan) =====

class TransformerBlock(nn.Module):
    """Tam Transformer bloğu (GPT-2 gibi ön-norm stili)."""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Rezidüel bağlantılı dikkat
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)
        
        # Rezidüel bağlantılı FFN
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)
        
        return x, attn_weights

print("TransformerBlock tanımlandı!")

## 2. Model Konfigürasyonu

Tüm model hiperparametrelerini bir araya getirmek için bir konfigürasyon veri sınıfı oluşturalım.

In [ ]:
@dataclass
class GPTConfig:
    """MiniGPT modeli için konfigürasyon."""
    vocab_size: int = 50257      # GPT-2 kelime dağarcığı boyutu
    max_seq_len: int = 1024      # Maksimum bağlam uzunluğu
    embed_dim: int = 768         # Gömme boyutu
    num_heads: int = 12          # Dikkat başlıklarının sayısı
    num_layers: int = 12         # Transformer bloklarının sayısı
    d_ff: int = 3072             # İleri beslemeli gizli boyut
    dropout: float = 0.1         # Dropout olasılığı

    def __post_init__(self):
        """Konfigürasyonu doğrula."""
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) num_heads ({self.num_heads}) ile bölünebilir olmalıdır"


# Farklı konfigürasyonları test et
print("GPT-2 Small (varsayılan):")
config = GPTConfig()
print(f"  Katmanlar: {config.num_layers}, Başlıklar: {config.num_heads}, Gömme: {config.embed_dim}")

print("\nMinik konfigürasyon (deneyler için):")
tiny_config = GPTConfig(embed_dim=64, num_heads=2, num_layers=2, d_ff=256)
print(f"  Katmanlar: {tiny_config.num_layers}, Başlıklar: {tiny_config.num_heads}, Gömme: {tiny_config.embed_dim}")

## 3. Tam MiniGPT Modeli

Şimdi Transformer bloklarını yığınlayarak ve ağırlık bağlaması ile dil modelleme başlığını ekleyerek tam modeli oluşturalım.

In [ ]:
class MiniGPT(nn.Module):
    """
    Minimal bir GPT tarzı dil modeli.
    Gömmeleri, Transformer bloklarını ve dil modelleme başlığını birleştirir.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Token ve Konum Gömmeleri =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Transformer Blokları =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Son LayerNorm =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Dil Modelleme Başlığı =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Ağırlık Bağlama =====
        self.lm_head.weight = self.token_embed.weight

        # Ağırlıkları başlat
        self._init_weights()

    def _init_weights(self):
        """Ağırlıkları küçük rastgele değerlerle başlat."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Model üzerinden ileri geçiş.

        Args:
            token_ids: Girdi token ID'leri (batch, seq)
            return_attention: Dikkat ağırlıklarını döndürüp döndürmeyeceği

        Returns:
            logits: Kelime dağarcığı skorları (batch, seq, vocab_size)
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # Gömmeler
        tok_emb = self.token_embed(token_ids)
        positions = torch.arange(seq, device=device)
        pos_emb = self.pos_embed(positions)
        x = self.dropout(tok_emb + pos_emb)

        # Nedensel maske
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # Transformer blokları
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # Son norm ve projeksiyon
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if return_attention:
            return logits, attention_weights
        return logits


print("MiniGPT sınıfı tanımlandı!")

In [ ]:
# Minik konfigürasyonla hızlı test
tiny_config = GPTConfig(embed_dim=64, num_heads=2, num_layers=2, d_ff=256)
model = MiniGPT(tiny_config)

# İleri geçişi test et
test_tokens = torch.randint(0, tiny_config.vocab_size, (2, 16))
logits = model(test_tokens)

print(f"Girdi şekli: {test_tokens.shape}")
print(f"Çıktı şekli: {logits.shape}")
print(f"\nBeklenen: (2, 16, {tiny_config.vocab_size})")

## 4. Sağlık Kontrolleri

5 temel testle modelimizin doğru şekilde bağlı olduğunu doğrulayalım.

In [ ]:
# ===== TEST 1: Şekil Doğrulama =====

def test_forward_shapes(config):
    """Model çıktı şekillerini doğrula."""
    model = MiniGPT(config)
    
    batch_size, seq_len = 2, 16
    token_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))
    
    logits = model(token_ids)
    
    expected_shape = (batch_size, seq_len, config.vocab_size)
    assert logits.shape == expected_shape, \
        f"Beklenen {expected_shape}, alınan {logits.shape}"
    
    print(f"Şekil kontrolü BAŞARILI: {logits.shape}")

test_forward_shapes(tiny_config)

In [ ]:
# ===== TEST 2: Parametre Sayısı =====

def count_parameters(model):
    """Toplam eğitilebilir parametreleri say."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def test_parameter_count(config):
    """Parametre sayısının makul olduğunu doğrula."""
    model = MiniGPT(config)
    total = count_parameters(model)
    print(f"Toplam parametreler: {total:,}")
    return total

# Minik konfigürasyonla test et
print("Minik model:")
test_parameter_count(tiny_config)

# Tam GPT-2 konfigürasyonuyla test et
print("\nGPT-2 Small:")
test_parameter_count(GPTConfig())

In [ ]:
# ===== TEST 3: Nedensel Maskeleme =====

def test_causal_masking():
    """Modelin gelecek tokenları göremediğini doğrula."""
    config = GPTConfig(
        num_layers=1,
        embed_dim=64,
        num_heads=2,
        d_ff=256,
        dropout=0.0  # Deterministik olması için dropout yok
    )
    model = MiniGPT(config)
    model.eval()

    # Aynı önek, farklı sonek
    tokens_a = torch.tensor([[100, 200, 300, 400]])
    tokens_b = torch.tensor([[100, 200, 300, 999]])  # Son token farklı

    with torch.no_grad():
        logits_a = model(tokens_a)
        logits_b = model(tokens_b)

    # 0, 1, 2 konumları AYNI olmalı
    for pos in range(3):
        assert torch.allclose(logits_a[0, pos], logits_b[0, pos], atol=1e-5), \
            f"Konum {pos} logitleri farklı!"

    # Konum 3 FARKLI olmalı
    assert not torch.allclose(logits_a[0, 3], logits_b[0, 3], atol=1e-5), \
        "Konum 3 logitleri farklı girdiye rağmen aynı!"

    print("Nedensel maskeleme BAŞARILI!")

test_causal_masking()

In [ ]:
# ===== TEST 4: Ağırlık Bağlama =====

def test_weight_tying():
    """Gömme ve lm_head'in ağırlıkları paylaştığını doğrula."""
    config = GPTConfig(embed_dim=64, num_heads=2, num_layers=2, d_ff=256)
    model = MiniGPT(config)

    # AYNI tensör olmalı
    assert model.lm_head.weight is model.token_embed.weight, \
        "Ağırlık bağlama başarısız: farklı tensörler!"

    # Birini değiştir, diğerinin değiştiğini kontrol et
    with torch.no_grad():
        original = model.token_embed.weight[0, 0].item()
        model.token_embed.weight[0, 0] = 999.0
        
        assert model.lm_head.weight[0, 0].item() == 999.0, \
            "Ağırlık bağlama başarısız: değişiklikler yayılmıyor!"
        
        model.token_embed.weight[0, 0] = original

    print("Ağırlık bağlama BAŞARILI!")

test_weight_tying()

In [ ]:
# ===== TEST 5: Gradyan Akışı =====

def test_gradient_flow():
    """Gradyanların tüm parametrelere ulaştığını doğrula."""
    config = GPTConfig(num_layers=2, embed_dim=64, num_heads=2, d_ff=256)
    model = MiniGPT(config)

    # İleri geçiş
    tokens = torch.randint(0, config.vocab_size, (1, 8))
    logits = model(tokens)

    # Geri geçiş
    loss = logits.sum()
    loss.backward()

    # Tüm parametrelerin gradyanları olduğunu kontrol et
    for name, param in model.named_parameters():
        assert param.grad is not None, f"{name} için gradyan yok"

    print("Gradyan akışı BAŞARILI!")

test_gradient_flow()

## 5. İlk İleri Geçişiniz

Gerçek metni modelimizden geçirelim ve rastgele ağırlıklarla ne çıktı verdiğini görelim.

In [ ]:
# Tokenizer'ı yükle
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Küçük model oluştur
config = GPTConfig(num_layers=2, embed_dim=256, num_heads=4, d_ff=1024)
model = MiniGPT(config)
model.eval()

# Bir prompt tokenize et
prompt = "The quick brown fox"
token_ids = tokenizer.encode(prompt, return_tensors="pt")

print(f"Prompt: '{prompt}'")
print(f"Token ID'leri: {token_ids}")
print(f"Tokenlar: {[tokenizer.decode([t]) for t in token_ids[0]]}")

In [ ]:
# İleri geçiş
with torch.no_grad():
    logits = model(token_ids)

print(f"Logit şekli: {logits.shape}")

# Sonraki token için en iyi 5 tahmini al
last_logits = logits[0, -1, :]
top_probs, top_indices = torch.softmax(last_logits, dim=-1).topk(5)

print(f"\n'{prompt}' sonrası en iyi 5 tahmin:")
for prob, idx in zip(top_probs, top_indices):
    token = tokenizer.decode([idx])
    print(f"  '{token}': {prob:.4f}")

print("\n(Rastgele tahminler - modelin ağırlıkları rastgele!)")

## 6. Önceden Eğitilmiş Ağırlıkları Yükleme

Şimdi heyecan verici kısım: gerçek GPT-2 ağırlıklarını MiniGPT'mize yükleyelim!

In [ ]:
def load_gpt2_weights(model, model_name="gpt2"):
    """
    Önceden eğitilmiş GPT-2 ağırlıklarını MiniGPT modelimize yükle.
    """
    from transformers import GPT2LMHeadModel

    print(f"'{model_name}' modelinden ağırlıklar yükleniyor...")

    # HuggingFace modelini yükle
    hf_model = GPT2LMHeadModel.from_pretrained(model_name)
    hf_state = hf_model.state_dict()

    # Bizim modelimizin state dict'i
    our_state = model.state_dict()

    # Gömmeleri kopyala
    our_state['token_embed.weight'].copy_(hf_state['transformer.wte.weight'])
    our_state['pos_embed.weight'].copy_(hf_state['transformer.wpe.weight'])

    # Her Transformer bloğunu kopyala
    for i in range(model.config.num_layers):
        # Layer normları
        our_state[f'blocks.{i}.ln1.weight'].copy_(
            hf_state[f'transformer.h.{i}.ln_1.weight'])
        our_state[f'blocks.{i}.ln1.bias'].copy_(
            hf_state[f'transformer.h.{i}.ln_1.bias'])
        our_state[f'blocks.{i}.ln2.weight'].copy_(
            hf_state[f'transformer.h.{i}.ln_2.weight'])
        our_state[f'blocks.{i}.ln2.bias'].copy_(
            hf_state[f'transformer.h.{i}.ln_2.bias'])

        # Dikkat (transpoze etmek gerekiyor!)
        our_state[f'blocks.{i}.attn.qkv_proj.weight'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_attn.weight'].T)
        our_state[f'blocks.{i}.attn.out_proj.weight'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_proj.weight'].T)

        # FFN (transpoze etmek gerekiyor!)
        our_state[f'blocks.{i}.ffn.fc1.weight'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_fc.weight'].T)
        our_state[f'blocks.{i}.ffn.fc1.bias'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_fc.bias'])
        our_state[f'blocks.{i}.ffn.fc2.weight'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_proj.weight'].T)
        our_state[f'blocks.{i}.ffn.fc2.bias'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_proj.bias'])

    # Son layer norm
    our_state['ln_f.weight'].copy_(hf_state['transformer.ln_f.weight'])
    our_state['ln_f.bias'].copy_(hf_state['transformer.ln_f.bias'])

    print("Ağırlıklar başarıyla yüklendi!")
    return model

In [ ]:
@torch.no_grad()
def generate_simple(model, tokenizer, prompt, max_new_tokens=20):
    """
    Açgözlü kod çözme kullanarak metin üret.
    """
    model.eval()
    device = next(model.parameters()).device

    # Prompt'u kodla
    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # Tokenları birer birer üret
    for _ in range(max_new_tokens):
        logits = model(token_ids)
        next_logits = logits[:, -1, :]
        next_token = next_logits.argmax(dim=-1, keepdim=True)
        token_ids = torch.cat([token_ids, next_token], dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(token_ids[0])

In [ ]:
# GPT-2 Small konfigürasyonuyla model oluştur
config = GPTConfig()  # Varsayılanlar GPT-2 Small ile eşleşir
model = MiniGPT(config)

# Önceden eğitilmiş ağırlıkları yükle
model = load_gpt2_weights(model, "gpt2")
model = model.to(device)

print(f"\nModel cihazda: {device}")
print(f"Parametreler: {count_parameters(model):,}")

In [ ]:
# Metin üret!
prompt = "The quick brown fox"
generated = generate_simple(model, tokenizer, prompt, max_new_tokens=30)

print(f"Prompt: '{prompt}'")
print(f"Üretilen: '{generated}'")

In [ ]:
# Daha fazla prompt dene!
prompts = [
    "Artificial intelligence will",
    "Once upon a time",
    "The capital of France is",
    "def fibonacci(n):"
]

for prompt in prompts:
    generated = generate_simple(model, tokenizer, prompt, max_new_tokens=20)
    print(f"'{prompt}' -> {generated}")
    print()

## 7. Rastgele ve Önceden Eğitilmiş Karşılaştırma

Rastgele ağırlıklar ile önceden eğitilmiş ağırlıkları dramatik bir şekilde karşılaştıralım.

In [ ]:
# Rastgele ağırlıklı model
model_random = MiniGPT(GPTConfig())
model_random = model_random.to(device)

prompt = "Artificial intelligence will"

print("=" * 50)
print("RASTGELE AĞIRLIKLAR:")
print(generate_simple(model_random, tokenizer, prompt, max_new_tokens=20))
print()
print("ÖNCEDEN EĞİTİLMİŞ AĞIRLIKLAR:")
print(generate_simple(model, tokenizer, prompt, max_new_tokens=20))
print("=" * 50)

print("\nAynı mimari. Aynı kod. Eğitim tüm farkı yaratıyor!")

## 8. Alıştırmalar

### Alıştırma 1: Minik Model

Belirli özelliklere sahip minik bir model oluşturun ve test edin.

In [ ]:
# KODUNUZ BURAYA
# Şu özelliklere sahip minik bir MiniGPT oluşturun:
# - vocab_size=500
# - max_seq_len=16
# - embed_dim=64
# - num_heads=2
# - num_layers=2
# - d_ff=256

# 1. Konfigürasyonu oluşturun
# 2. Modeli oluşturun
# 3. Rastgele tokenlar üzerinde ileri geçiş yapın
# 4. Çıktı şeklini doğrulayın
# 5. Parametreleri sayın

### Alıştırma 2: Sıcaklık Keşfi

Sıcaklık ölçeklemeyi kullanmak için üretimi değiştirin.

In [ ]:
# KODUNUZ BURAYA
# generate_simple'ı bir sıcaklık parametresi kabul edecek şekilde değiştirin:
# next_logits = next_logits / temperature
# 
# Şu sıcaklıkları deneyin: 0.5, 1.0, 1.5
# Çıktı nasıl değişiyor?

### Alıştırma 3: Dikkat Görselleştirme

Önceden eğitilmiş modeldeki dikkat desenlerini görselleştirin.

In [ ]:
# KODUNUZ BURAYA
# 1. Modeli return_attention=True ile çalıştırın
# 2. Son bloktan dikkat ağırlıklarını alın
# 3. Başlık 0 için bir ısı haritası çizin
# İpucu: matplotlib.pyplot.imshow() kullanın

## Özet

**Oluşturduklarımız:**

1. **GPTConfig**: Temiz konfigürasyon deseni
2. **MiniGPT**: Ağırlık bağlamalı tam dil modeli
3. **Sağlık kontrolleri**: Doğruluğu kontrol etmek için 5 test
4. **Ağırlık yükleme**: GPT-2 ağırlıklarını aktarma
5. **Metin üretimi**: Açgözlü kod çözme

**Temel kavramlar:**

- Katmanları yığınlamak için `nn.ModuleList`
- Ağırlık bağlama 38M parametre tasarrufu sağlar
- LM başlığı: `(batch, seq, embed_dim)` → `(batch, seq, vocab_size)`
- `model.train()` ve `model.eval()`

**Sonraki:** Bölüm 12, bu modeli sıfırdan eğitmeyi öğretecek!